# Team B — Week 5: Cross-Model Profiling (Llama-2 7B & 13B)
**IBM HPML Uncertainty-Aware Inference Project — Systems Lead**

This notebook extends Team B's Mistral-7B profiling harness to Teams A and C's models:
- **Llama-2 7B** (Team A) — all 5 PTQ configs
- **Llama-2 13B** (Team C) — all 5 PTQ configs

**Deliverables (Week 5 Systems Lead):**
1. Per-config PyTorch Profiler JSONs for both model families
2. Cross-model Roofline plots (all 15 model×config pairs on one figure)
3. Compute-bound vs. memory-bound classification table per precision level
4. W&B logging into `UAI_Project`

> 💡 Load the cached Mistral-7B results from `/content/drive/MyDrive/uai_results/TeamB/profiler_results`
> and add them to the cross-model figure without re-running.


## 0 · Setup

In [ ]:
# ── Install dependencies (Colab A100) ──────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("transformers>=4.40.0", "accelerate>=0.28.0")
pip("gptqmodel>=1.2.0")          # GPTQ loading (handles Marlin kernel auto-detect)
pip("autoawq>=0.2.5")            # AWQ loading
pip("bitsandbytes>=0.43.0")      # NF4
pip("wandb")
pip("matplotlib", "numpy", "pandas")
print("✓ Dependencies installed")


In [ ]:
import os, gc, json, time, logging, warnings
from pathlib import Path
import numpy as np
import torch

# ── CUPTI must be on LD_LIBRARY_PATH before torch.profiler is used ──────────
_CUPTI = ["/usr/local/cuda/extras/CUPTI/lib64", "/usr/local/cuda/lib64"]
_ld    = os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = ":".join(p for p in _CUPTI if os.path.isdir(p)) + (":"+ _ld if _ld else "")
os.environ["KINETO_USE_DAEMON"] = "0"

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("week5_profiler")

# ── Auth ────────────────────────────────────────────────────────────────────
from google.colab import userdata
HF_TOKEN   = userdata.get("HF_TOKEN")
WANDB_KEY  = userdata.get("WANDB_API_KEY")

os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import login as hf_login
hf_login(token=HF_TOKEN, add_to_git_credential=False)

import wandb
wandb.login(key=WANDB_KEY)
print(f"✓ GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


In [ ]:
# ── Mount Drive and define output paths ────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT        = Path("/content/drive/MyDrive/uai_results")
MISTRAL_PROF_DIR  = DRIVE_ROOT / "TeamB"  / "profiler_results"
LLAMA7B_PROF_DIR  = DRIVE_ROOT / "TeamA"  / "profiler_results"
LLAMA13B_PROF_DIR = DRIVE_ROOT / "TeamC"  / "profiler_results"
CROSS_OUT_DIR     = DRIVE_ROOT / "week5"  / "cross_model_profiling"

for d in [LLAMA7B_PROF_DIR, LLAMA13B_PROF_DIR, CROSS_OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("✓ Drive mounted — output dirs ready")


## 1 · Extended Model Registries

In [ ]:
# ── Llama-2 7B (Team A) — 5 PTQ configs ────────────────────────────────────
# TheBloke GPTQ checkpoints are the standard community source; they match
# exactly what Team A uses for calibration eval.
LLAMA7B_REGISTRY = {
    "llama2-7b-fp16": {
        "hf_id":       "meta-llama/Llama-2-7b-chat-hf",
        "quant_type":  "fp16",
        "bits":        16,
        "description": "Llama-2 7B FP16",
        "family":      "llama2-7b",
    },
    "llama2-7b-gptq-int8": {
        "hf_id":          "TheBloke/Llama-2-7B-Chat-GPTQ",
        "quant_type":     "gptq",
        "bits":           8,
        "gptq_revision":  "gptq-8bit-128g-actorder_True",
        "description":    "Llama-2 7B GPTQ INT8",
        "family":         "llama2-7b",
    },
    "llama2-7b-gptq-int4": {
        "hf_id":          "TheBloke/Llama-2-7B-Chat-GPTQ",
        "quant_type":     "gptq",
        "bits":           4,
        "gptq_revision":  "main",
        "description":    "Llama-2 7B GPTQ INT4",
        "family":         "llama2-7b",
    },
    "llama2-7b-awq-int4": {
        "hf_id":       "TheBloke/Llama-2-7B-Chat-AWQ",
        "quant_type":  "awq",
        "bits":        4,
        "description": "Llama-2 7B AWQ INT4",
        "family":      "llama2-7b",
    },
    "llama2-7b-nf4": {
        "hf_id":       "meta-llama/Llama-2-7b-chat-hf",
        "quant_type":  "nf4",
        "bits":        4,
        "description": "Llama-2 7B NF4",
        "family":      "llama2-7b",
    },
}

# ── Llama-2 13B (Team C) — 5 PTQ configs ────────────────────────────────────
LLAMA13B_REGISTRY = {
    "llama2-13b-fp16": {
        "hf_id":       "meta-llama/Llama-2-13b-chat-hf",
        "quant_type":  "fp16",
        "bits":        16,
        "description": "Llama-2 13B FP16",
        "family":      "llama2-13b",
    },
    "llama2-13b-gptq-int8": {
        "hf_id":          "TheBloke/Llama-2-13B-chat-GPTQ",
        "quant_type":     "gptq",
        "bits":           8,
        "gptq_revision":  "gptq-8bit-128g-actorder_True",
        "description":    "Llama-2 13B GPTQ INT8",
        "family":         "llama2-13b",
    },
    "llama2-13b-gptq-int4": {
        "hf_id":          "TheBloke/Llama-2-13B-chat-GPTQ",
        "quant_type":     "gptq",
        "bits":           4,
        "gptq_revision":  "main",
        "description":    "Llama-2 13B GPTQ INT4",
        "family":         "llama2-13b",
    },
    "llama2-13b-awq-int4": {
        "hf_id":       "TheBloke/Llama-2-13B-chat-AWQ",
        "quant_type":  "awq",
        "bits":        4,
        "description": "Llama-2 13B AWQ INT4",
        "family":      "llama2-13b",
    },
    "llama2-13b-nf4": {
        "hf_id":       "meta-llama/Llama-2-13b-chat-hf",
        "quant_type":  "nf4",
        "bits":        4,
        "description": "Llama-2 13B NF4",
        "family":      "llama2-13b",
    },
}

print("Llama-2 7B configs :", list(LLAMA7B_REGISTRY))
print("Llama-2 13B configs:", list(LLAMA13B_REGISTRY))


## 2 · Model Loader

In [ ]:
# Reuses Team B's load_model_for_profiling pattern — unchanged logic,
# parameterised over whichever registry is passed in.

def load_model_for_profiling(config_key: str, registry: dict):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    cfg        = registry[config_key]
    hf_id      = cfg["hf_id"]
    quant_type = cfg["quant_type"]
    bits       = cfg["bits"]
    log.info(f"Loading [{config_key}]  hf_id={hf_id}  quant={quant_type}  bits={bits}")

    tok_kwargs = {"token": HF_TOKEN, "trust_remote_code": True}
    tokenizer  = AutoTokenizer.from_pretrained(hf_id, **tok_kwargs)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if quant_type == "fp16":
        model = AutoModelForCausalLM.from_pretrained(
            hf_id, torch_dtype=torch.float16,
            device_map="auto", **tok_kwargs)

    elif quant_type == "gptq":
        from gptqmodel import GPTQModel
        revision = cfg.get("gptq_revision", "main")
        model = GPTQModel.from_quantized(
            hf_id, revision=revision, device="cuda:0", **tok_kwargs)

    elif quant_type == "awq":
        from awq import AutoAWQForCausalLM
        model = AutoAWQForCausalLM.from_quantized(
            hf_id, fuse_layers=False, **tok_kwargs)

    elif quant_type == "nf4":
        from transformers import BitsAndBytesConfig
        bnb = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
        model = AutoModelForCausalLM.from_pretrained(
            hf_id, quantization_config=bnb,
            device_map="auto", **tok_kwargs)
    else:
        raise ValueError(f"Unknown quant_type '{quant_type}'")

    model.eval()
    mem_gb = torch.cuda.memory_allocated() / 1e9
    exp_gb = bits / 16 * 14.0   # rough expected size relative to fp16 baseline
    flag   = "  ← unexpected size?" if mem_gb > exp_gb * 1.6 else ""
    log.info(f"  GPU mem after load: {mem_gb:.2f} GB  (expected ~{exp_gb:.1f} GB){flag}")
    return model, tokenizer


## 3 · Profiling Harness

In [ ]:
# Direct port of Team B's profile_inference_fixed() with:
#   ✓ cuda.synchronize() inside profiler context
#   ✓ CUDA time attribute fallback chain (handles torch >= 2.1)
#   ✓ profile_memory=False  (avoids kineto_results=None on some builds)
#   ✓ with_modules=False    (torch.fx tracing errors on HF models)

from torch.profiler import ProfilerActivity, profile, record_function

@torch.no_grad()
def profile_inference(
    model, tokenizer, config_key: str, registry: dict,
    prompt: str = "Explain the role of attention in transformer models.",
    n_tokens: int = 50,
    warmup_steps: int = 3,
    profile_steps: int = 5,
    output_dir: Path = None,
    wandb_run=None,
) -> dict:
    cfg = registry[config_key]
    device = (torch.device("cuda", torch.cuda.current_device())
              if torch.cuda.is_available() else torch.device("cpu"))

    inputs    = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]
    attn_mask = inputs["attention_mask"]
    gen_kwargs = dict(
        input_ids=input_ids, attention_mask=attn_mask,
        max_new_tokens=n_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    # ── Warmup ──────────────────────────────────────────────────────────────
    log.info(f"[{config_key}] Warmup ({warmup_steps} steps)…")
    for _ in range(warmup_steps):
        model.generate(**gen_kwargs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    # ── Profiling loop ───────────────────────────────────────────────────────
    timing_ms = []
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    log.info(f"[{config_key}] Profiling ({profile_steps} steps)…")
    with profile(
        activities=activities,
        record_shapes=True, profile_memory=False,
        with_flops=True, with_stack=False, with_modules=False,
    ) as prof:
        for step in range(profile_steps):
            with record_function(f"inference_step_{step}"):
                t0 = time.perf_counter()
                model.generate(**gen_kwargs)
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                timing_ms.append((time.perf_counter() - t0) * 1000)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    if output_dir:
        trace_path = Path(output_dir) / f"{config_key}_chrome.json"
        try:
            prof.export_chrome_trace(str(trace_path))
        except AttributeError:
            log.warning("  Chrome trace skipped (kineto_results=None)")

    # ── Aggregate ────────────────────────────────────────────────────────────
    avg_ms  = float(np.mean(timing_ms))
    tps     = n_tokens / (avg_ms / 1000) if avg_ms > 0 else 0.0
    mem_gb  = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0

    key_avgs = prof.key_averages()

    def _cuda_t(e):
        for attr in ("cuda_time_total","self_cuda_time_total","device_time_total"):
            v = getattr(e, attr, None)
            if v is not None: return v
        return 0.0

    sample = [_cuda_t(e) for e in key_avgs]
    has_cuda = any(v > 0 for v in sample)

    SPAN_PFX = ("inference_step_",)

    if has_cuda:
        cuda_events   = [e for e in key_avgs if _cuda_t(e) > 0]
        total_cuda_us = sum(_cuda_t(e) for e in cuda_events)
        real_events   = [e for e in cuda_events if not any(e.key.startswith(p) for p in SPAN_PFX)]
        top_raw       = sorted(real_events, key=_cuda_t, reverse=True)[:10]
    else:
        log.warning(f"  [{config_key}] No GPU kernel times — falling back to cpu_time_total")
        cuda_events   = [e for e in key_avgs if e.cpu_time_total > 0]
        total_cuda_us = sum(e.cpu_time_total for e in cuda_events)
        real_events   = [e for e in cuda_events if not any(e.key.startswith(p) for p in SPAN_PFX)]
        top_raw       = sorted(real_events, key=lambda e: e.cpu_time_total, reverse=True)[:10]

    avg_cuda_us      = total_cuda_us / profile_steps
    total_flops_all  = sum(getattr(e,"flops",0) or 0 for e in key_avgs)
    avg_flops        = total_flops_all / profile_steps
    arith_intensity  = avg_flops / (mem_gb * 1e9) if mem_gb > 0 else 0.0

    def kt(e): return (_cuda_t(e) if has_cuda else e.cpu_time_total)

    result = {
        "config_key":  config_key,
        "hf_id":       cfg["hf_id"],
        "family":      cfg.get("family", "unknown"),
        "quant_type":  cfg["quant_type"],
        "bits":        cfg["bits"],
        "description": cfg["description"],
        "kineto_cuda": has_cuda,
        "timing": {
            "total_inference_ms": avg_ms,
            "tokens_per_second":  tps,
            "all_step_ms":        timing_ms,
        },
        "memory": {"peak_gpu_gb": mem_gb},
        "compute": {
            "avg_cuda_ms":         avg_cuda_us / 1e3,
            "avg_flops_per_step":  float(avg_flops),
            "arithmetic_intensity":float(arith_intensity),
        },
        "top_kernels": [
            {
                "name":         e.key,
                "cuda_time_ms": kt(e) / profile_steps / 1e3,
                "pct":          kt(e) / total_cuda_us * 100 if total_cuda_us else 0,
                "calls":        max(e.count // profile_steps, 1) if e.count > 0 else 0,
            }
            for e in top_raw
        ],
    }

    log.info(f"  [{config_key}] {avg_ms:.0f} ms  |  {tps:.1f} tok/s  |  "
             f"{mem_gb:.2f} GB peak  |  AI={arith_intensity:.1f}")

    if output_dir:
        json_path = Path(output_dir) / f"{config_key}_profile.json"
        with open(json_path, "w") as f:
            json.dump(result, f, indent=2)
        log.info(f"  Saved → {json_path}")

    if wandb_run:
        wandb_run.log({
            "profiler/avg_inference_ms":  avg_ms,
            "profiler/tokens_per_second": tps,
            "profiler/peak_gpu_gb":       mem_gb,
            "profiler/avg_cuda_ms":       avg_cuda_us / 1e3,
            "profiler/arithmetic_intensity": arith_intensity,
        })

    return result


## 4 · Run Profiling — Llama-2 7B (Team A)

In [ ]:
# Profiles each Llama-2 7B config. Skip any that already have a cached JSON.
# Each model is freed from GPU memory after profiling so the next can load.

llama7b_results = {}
FORCE = False   # set True to re-profile even if JSON exists

for config_key in LLAMA7B_REGISTRY:
    json_path = LLAMA7B_PROF_DIR / f"{config_key}_profile.json"
    if json_path.exists() and not FORCE:
        with open(json_path) as f:
            llama7b_results[config_key] = json.load(f)
        log.info(f"[{config_key}] Loaded cached — skipping profiling")
        continue

    if not torch.cuda.is_available():
        log.warning("No GPU — skipping"); continue

    wandb_run = wandb.init(
        project="UAI_Project", entity="Uncertainty_Aware_Inference_Lab",
        name=f"teamB_week5_{config_key}_profiler",
        config={**LLAMA7B_REGISTRY[config_key], "experiment": "week5_cross_model_profiling"},
        reinit=True,
    )

    model, tokenizer = load_model_for_profiling(config_key, LLAMA7B_REGISTRY)
    llama7b_results[config_key] = profile_inference(
        model, tokenizer, config_key, LLAMA7B_REGISTRY,
        output_dir=LLAMA7B_PROF_DIR, wandb_run=wandb_run,
    )
    wandb_run.finish()

    del model; gc.collect(); torch.cuda.empty_cache()
    log.info(f"  [{config_key}] GPU freed")

print(f"\nLlama-2 7B profiling complete. Results: {list(llama7b_results)}")


## 5 · Run Profiling — Llama-2 13B (Team C)

In [ ]:
# Same pattern as above for the 13B model.
# The 13B FP16 model requires ~26 GB VRAM — run on A100-80GB.
# If you're on a 40 GB instance, skip FP16 and use the quantized configs only.

GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
SKIP_13B_FP16 = GPU_VRAM_GB < 40

if SKIP_13B_FP16:
    print(f"⚠  Only {GPU_VRAM_GB:.0f} GB VRAM detected — skipping Llama-2 13B FP16")

llama13b_results = {}

for config_key in LLAMA13B_REGISTRY:
    if SKIP_13B_FP16 and "fp16" in config_key:
        log.info(f"[{config_key}] Skipped (insufficient VRAM)")
        continue

    json_path = LLAMA13B_PROF_DIR / f"{config_key}_profile.json"
    if json_path.exists() and not FORCE:
        with open(json_path) as f:
            llama13b_results[config_key] = json.load(f)
        log.info(f"[{config_key}] Loaded cached — skipping profiling")
        continue

    wandb_run = wandb.init(
        project="UAI_Project", entity="Uncertainty_Aware_Inference_Lab",
        name=f"teamB_week5_{config_key}_profiler",
        config={**LLAMA13B_REGISTRY[config_key], "experiment": "week5_cross_model_profiling"},
        reinit=True,
    )

    model, tokenizer = load_model_for_profiling(config_key, LLAMA13B_REGISTRY)
    llama13b_results[config_key] = profile_inference(
        model, tokenizer, config_key, LLAMA13B_REGISTRY,
        output_dir=LLAMA13B_PROF_DIR, wandb_run=wandb_run,
    )
    wandb_run.finish()
    del model; gc.collect(); torch.cuda.empty_cache()

print(f"\nLlama-2 13B profiling complete. Results: {list(llama13b_results)}")


## 6 · Load Mistral-7B Cached Results

In [ ]:
# Loads Week 1-4 Mistral-7B profiler results.
# These were produced by teamb_profiler.ipynb and saved to Drive.

MISTRAL_REGISTRY_META = {
    "mistral-7b-fp16":      {"family":"mistral-7b","bits":16,"quant_type":"fp16",   "description":"Mistral 7B FP16"},
    "mistral-7b-gptq-int8": {"family":"mistral-7b","bits":8, "quant_type":"gptq",   "description":"Mistral 7B GPTQ INT8"},
    "mistral-7b-gptq-int4": {"family":"mistral-7b","bits":4, "quant_type":"gptq",   "description":"Mistral 7B GPTQ INT4"},
    "mistral-7b-awq-int4":  {"family":"mistral-7b","bits":4, "quant_type":"awq",    "description":"Mistral 7B AWQ INT4"},
    "mistral-7b-nf4":       {"family":"mistral-7b","bits":4, "quant_type":"nf4",    "description":"Mistral 7B NF4"},
}

mistral_results = {}
for config_key, meta in MISTRAL_REGISTRY_META.items():
    json_path = MISTRAL_PROF_DIR / f"{config_key}_profile.json"
    if json_path.exists():
        with open(json_path) as f:
            r = json.load(f)
        # Backfill family/description if not in older JSON format
        r.setdefault("family",      meta["family"])
        r.setdefault("description", meta["description"])
        mistral_results[config_key] = r
        log.info(f"[{config_key}] Loaded from {json_path}")
    else:
        log.warning(f"[{config_key}] Not found at {json_path} — will be missing from cross-model plots")

print(f"\nMistral-7B results loaded: {list(mistral_results)}")


## 7 · Compute-Bound vs. Memory-Bound Classification

In [ ]:
# ── A100-80GB hardware parameters for Roofline ──────────────────────────────
# Source: NVIDIA A100 Datasheet (https://www.nvidia.com/en-us/data-center/a100/)
A100_PEAK_FP16_TFLOPS = 312.0     # Tensor Core peak (TF32 matmul on FP16 data)
A100_MEM_BW_TB_S      = 2.0       # 2 TB/s HBM2e bandwidth
A100_RIDGE_POINT      = (A100_PEAK_FP16_TFLOPS * 1e12) / (A100_MEM_BW_TB_S * 1e12)
# Ridge point = 312 TFLOPS / 2 TB/s = 156 FLOPs/byte

print(f"A100 ridge point: {A100_RIDGE_POINT:.0f} FLOPs/byte")
print("  Arithmetic intensity > ridge point  →  compute-bound")
print("  Arithmetic intensity < ridge point  →  memory-bound")

def classify_bound(ai: float) -> str:
    """Returns 'Compute-bound', 'Memory-bound', or 'Near ridge'."""
    if ai == 0:
        return "Unknown (flops=0)"
    if   ai > A100_RIDGE_POINT * 1.1:  return "Compute-bound"
    elif ai < A100_RIDGE_POINT * 0.9:  return "Memory-bound"
    else:                               return "Near ridge"

# Build unified results dict for all three model families
all_results = {**mistral_results, **llama7b_results, **llama13b_results}

import pandas as pd

rows = []
for key, r in all_results.items():
    ai   = r["compute"]["arithmetic_intensity"]
    rows.append({
        "Config":        r.get("description", key),
        "Family":        r.get("family", "?"),
        "Quant":         r["quant_type"],
        "Bits":          r["bits"],
        "Infer (ms)":    round(r["timing"]["total_inference_ms"],  1),
        "Tok/s":         round(r["timing"]["tokens_per_second"],   1),
        "Peak GPU (GB)": round(r["memory"]["peak_gpu_gb"],         2),
        "CUDA (ms)":     round(r["compute"]["avg_cuda_ms"],        1),
        "AI (FLOPs/B)":  round(ai,                                  1),
        "Bound":         classify_bound(ai),
        "Kineto":        "✓" if r.get("kineto_cuda") else "~",
    })

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 35)
display(df.sort_values(["Family","Bits"]))


In [ ]:
# ── Per-precision summary: compute-bound vs memory-bound ────────────────────
# This is the key Week 5 deliverable: which configs are compute-bound?

print("\n" + "="*70)
print(" COMPUTE vs MEMORY BOUND — PER PRECISION LEVEL")
print("="*70)

for bits in [16, 8, 4]:
    subset = df[df["Bits"] == bits]
    if subset.empty: continue
    bound_counts = subset["Bound"].value_counts()
    print(f"\n  {bits}-bit configs ({len(subset)} total across all families):")
    for bound, count in bound_counts.items():
        configs = subset[subset["Bound"]==bound]["Config"].tolist()
        print(f"    {bound:<18}: {count}  →  {configs}")

print()
print("Key insight: Memory-bound configs waste GPU arithmetic capacity.")
print("Compute-bound configs are GPU-efficient but may have higher latency.")


## 8 · Cross-Model Roofline Plot

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# ── Roofline curve ────────────────────────────────────────────────────────────
ai_range   = np.logspace(-1, 4, 500)
roof_tflops = np.minimum(
    A100_PEAK_FP16_TFLOPS,
    A100_MEM_BW_TB_S * ai_range / 1e3   # FLOPs/byte × TB/s → TFLOPS
)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 7))
ax.set_xscale("log"); ax.set_yscale("log")

# Draw roofline
ax.plot(ai_range, roof_tflops, "k-", lw=2.5, label="A100 Roofline (FP16, 312 TFLOPS)")
ax.axvline(A100_RIDGE_POINT, color="gray", linestyle="--", lw=1.2, alpha=0.7)
ax.text(A100_RIDGE_POINT * 1.05, 0.35, f"Ridge: {A100_RIDGE_POINT:.0f} FLOPs/B",
        color="gray", fontsize=9, va="bottom")

# ── Markers per family and quant type ─────────────────────────────────────────
FAMILY_MARKERS = {"mistral-7b": "o", "llama2-7b": "s", "llama2-13b": "^"}
QUANT_COLORS   = {
    "fp16": "#2196F3",   # blue
    "gptq": "#F44336",   # red  (covers int4 and int8)
    "awq":  "#4CAF50",   # green
    "nf4":  "#FF9800",   # orange
}
BITS_SIZE = {16: 180, 8: 120, 4: 80}

for key, r in all_results.items():
    ai   = r["compute"]["arithmetic_intensity"]
    if ai == 0: continue   # skip if flops counter returned 0 (AWQ kernel)

    # Estimate achievable TFLOPS from profiler timing
    flops  = r["compute"]["avg_flops_per_step"]
    cuda_s = r["compute"]["avg_cuda_ms"] / 1e3
    if flops > 0 and cuda_s > 0:
        achieved_tflops = flops / cuda_s / 1e12
    else:
        achieved_tflops = A100_PEAK_FP16_TFLOPS * 0.1   # placeholder

    family = r.get("family", "mistral-7b")
    qt     = r["quant_type"]
    bits   = r["bits"]

    marker = FAMILY_MARKERS.get(family, "D")
    color  = QUANT_COLORS.get(qt, "#9C27B0")
    size   = BITS_SIZE.get(bits, 80)

    ax.scatter(ai, achieved_tflops, marker=marker, s=size, color=color,
               edgecolors="black", linewidths=0.6, zorder=5, alpha=0.85)
    ax.annotate(r.get("description",""), xy=(ai, achieved_tflops),
                xytext=(5, 3), textcoords="offset points",
                fontsize=7.5, alpha=0.85)

# ── Legend ─────────────────────────────────────────────────────────────────────
family_handles = [
    Line2D([0],[0], marker=m, color="w", markerfacecolor="gray",
           markeredgecolor="black", markersize=8, label=f)
    for f, m in FAMILY_MARKERS.items()
]
quant_handles = [
    mpatches.Patch(color=c, label=qt)
    for qt, c in QUANT_COLORS.items()
]
bits_handles = [
    Line2D([0],[0], marker="o", color="w", markerfacecolor="gray",
           markeredgecolor="black", markersize=np.sqrt(s)*0.6, label=f"{b}-bit")
    for b, s in BITS_SIZE.items()
]

leg1 = ax.legend(handles=family_handles, title="Model family", loc="upper left",  fontsize=8)
leg2 = ax.legend(handles=quant_handles,  title="Quant method", loc="lower right", fontsize=8)
leg3 = ax.legend(handles=bits_handles,   title="Bit-width",    loc="upper right", fontsize=8)
ax.add_artist(leg1); ax.add_artist(leg2)

ax.set_xlabel("Arithmetic Intensity (FLOPs / byte)", fontsize=12)
ax.set_ylabel("Achieved Performance (TFLOPS)",        fontsize=12)
ax.set_title("Cross-Model Roofline — Mistral-7B, Llama-2 7B & 13B (A100-80GB, FP16 roof)",
             fontsize=12, fontweight="bold")
ax.grid(True, which="both", alpha=0.3)
ax.set_xlim(0.1, 5000)
ax.set_ylim(0.01, 1000)

plt.tight_layout()
fig_path = CROSS_OUT_DIR / "crossmodel_roofline.png"
fig.savefig(fig_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved → {fig_path}")


## 9 · Top-Kernel Heatmap — GPTQ vs AWQ vs NF4

In [ ]:
# Build a heatmap showing which kernels dominate each config.
# Columns = config keys; rows = top kernel names; values = % of CUDA time.

import re

def shorten_kernel(name: str, maxlen: int = 30) -> str:
    """Strip template parameters and long path prefixes for display."""
    name = re.sub(r"<[^>]*>", "", name)   # strip template args
    name = name.split("::")[-1]           # keep last scope element
    return name[:maxlen]

# Gather all unique kernel names across all configs
kernel_names = set()
for r in all_results.values():
    for k in r.get("top_kernels", []):
        kernel_names.add(shorten_kernel(k["name"]))

# Build matrix: rows=kernels, cols=configs
configs_ordered = [
    k for k in [
        "mistral-7b-fp16","mistral-7b-gptq-int8","mistral-7b-gptq-int4",
        "mistral-7b-awq-int4","mistral-7b-nf4",
        "llama2-7b-fp16","llama2-7b-gptq-int8","llama2-7b-gptq-int4",
        "llama2-7b-awq-int4","llama2-7b-nf4",
        "llama2-13b-fp16","llama2-13b-gptq-int8","llama2-13b-gptq-int4",
        "llama2-13b-awq-int4","llama2-13b-nf4",
    ] if k in all_results
]

labels_x = [all_results[k]["description"] for k in configs_ordered]

# Pick top 10 kernels globally by max pct across any config
kernel_max_pct = {}
for kname in kernel_names:
    vals = []
    for r in all_results.values():
        for k in r.get("top_kernels", []):
            if shorten_kernel(k["name"]) == kname:
                vals.append(k["pct"])
    kernel_max_pct[kname] = max(vals) if vals else 0
top_kernels_global = sorted(kernel_max_pct, key=kernel_max_pct.get, reverse=True)[:12]

matrix = np.zeros((len(top_kernels_global), len(configs_ordered)))
for j, ckey in enumerate(configs_ordered):
    for k in all_results[ckey].get("top_kernels", []):
        sname = shorten_kernel(k["name"])
        if sname in top_kernels_global:
            i = top_kernels_global.index(sname)
            matrix[i, j] += k["pct"]

fig2, ax2 = plt.subplots(figsize=(max(14, len(configs_ordered)*0.9), 7))
im = ax2.imshow(matrix, aspect="auto", cmap="YlOrRd", vmin=0, vmax=15)
plt.colorbar(im, ax=ax2, label="% of CUDA time")

ax2.set_xticks(range(len(configs_ordered))); ax2.set_xticklabels(labels_x, rotation=35, ha="right", fontsize=8)
ax2.set_yticks(range(len(top_kernels_global))); ax2.set_yticklabels(top_kernels_global, fontsize=8)
ax2.set_title("Top-Kernel CUDA Time Distribution — All Models × Configs", fontsize=11, fontweight="bold")

for i in range(len(top_kernels_global)):
    for j in range(len(configs_ordered)):
        val = matrix[i, j]
        if val > 0.5:
            ax2.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=7,
                     color="white" if val > 8 else "black")

plt.tight_layout()
fig2_path = CROSS_OUT_DIR / "crossmodel_kernel_heatmap.png"
fig2.savefig(fig2_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved → {fig2_path}")


## 10 · Save Cross-Model Summary JSON

In [ ]:
# Merge everything into a single summary file for Team C's Pareto analysis
# and the report.

summary = {
    "hardware": {
        "gpu":                "NVIDIA A100",
        "peak_fp16_tflops":   A100_PEAK_FP16_TFLOPS,
        "memory_bandwidth_tb_s": A100_MEM_BW_TB_S,
        "ridge_point_flops_per_byte": A100_RIDGE_POINT,
    },
    "configs": {},
}

for key, r in all_results.items():
    ai = r["compute"]["arithmetic_intensity"]
    summary["configs"][key] = {
        "family":      r.get("family", "unknown"),
        "description": r.get("description", key),
        "quant_type":  r["quant_type"],
        "bits":        r["bits"],
        "timing_ms":   r["timing"]["total_inference_ms"],
        "tok_per_sec": r["timing"]["tokens_per_second"],
        "peak_gpu_gb": r["memory"]["peak_gpu_gb"],
        "avg_cuda_ms": r["compute"]["avg_cuda_ms"],
        "arithmetic_intensity": ai,
        "bound_classification": classify_bound(ai),
        "top_kernel": r["top_kernels"][0]["name"] if r.get("top_kernels") else "N/A",
    }

out_json = CROSS_OUT_DIR / "crossmodel_profiling_summary.json"
with open(out_json, "w") as f:
    json.dump(summary, f, indent=2)
print(f"Cross-model summary saved → {out_json}")
print(f"\nAll Week 5 outputs in: {CROSS_OUT_DIR}")
